# Bid Ladder via Two-Way Stochastic Optimization

In [1]:
# https://edoc.hu-berlin.de/server/api/core/bitstreams/3a576fe5-35b5-4772-8a46-5d21caa89d6e/content

In [1]:
import sys
sys.path.append('../../')

import numpy as np
from scipy.optimize import minimize
import pandas as pd
import matplotlib.pyplot as plt
from docplex.mp.model import Model
from datetime import datetime, timedelta
from optimization.optim1.StochasticOptim import BiddingOptimization

In [2]:
from docplex.mp.environment import Environment

env = Environment()
env.cplex_path = r"C:\Program Files\IBM\ILOG\CPLEX_Studio\version\cplex\bin\x64_win64\cplex.exe"

In [3]:
def generate_dates(start_date, end_date):
    """
    Generate a list of dates in 'YYYY-MM-DD' format between the start_date and end_date (inclusive).

    :param start_date: The start date as a string in 'YYYY-MM-DD' format.
    :param end_date: The end date as a string in 'YYYY-MM-DD' format.
    :return: A list of dates in 'YYYY-MM-DD' format.
    """
    # Parse the start and end dates
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    
    # Generate the dates
    date_list = [(start + timedelta(days=i)).strftime("%Y-%m-%d") 
                 for i in range((end - start).days + 1)]
    return date_list

# Example usage
dates = generate_dates("2024-09-17", "2024-10-16")
print(dates)


['2024-09-17', '2024-09-18', '2024-09-19', '2024-09-20', '2024-09-21', '2024-09-22', '2024-09-23', '2024-09-24', '2024-09-25', '2024-09-26', '2024-09-27', '2024-09-28', '2024-09-29', '2024-09-30', '2024-10-01', '2024-10-02', '2024-10-03', '2024-10-04', '2024-10-05', '2024-10-06', '2024-10-07', '2024-10-08', '2024-10-09', '2024-10-10', '2024-10-11', '2024-10-12', '2024-10-13', '2024-10-14', '2024-10-15', '2024-10-16']


## Helper Functions Configuration to Run Optimizer With Different Setups

In [4]:
def plot_bidding(DA_bids_optimal, da_prices, date, risk_aversion=0, cvar=0.95):
    """
    Plot Day-Ahead Bids and Prices for each scenario.

    :param DA_bids_optimal: A 2D array of optimal Day-Ahead bids (shape: [scenarios, hours]).
    :param da_prices: A 2D array of Day-Ahead prices for each scenario (shape: [scenarios, hours]).
    :param date: The date for which the plot is created (string format 'YYYY-MM-DD').
    :param risk_aversion: The risk aversion coefficient used in optimization.
    :param cvar: The CVaR level used in optimization.
    """
    # Loop through each scenario to plot individual charts
    for scenario in range(DA_bids_optimal.shape[0]):
        fig, ax1 = plt.subplots(figsize=(12, 4))
        
        # Plot Day-Ahead Bids on primary y-axis
        ax1.scatter(
            range(1, 25), 
            DA_bids_optimal[scenario], 
            edgecolor='green', 
            facecolor='none', 
            marker='^', 
            s=80, 
            label="DA Bids"
        )
        ax1.set_xlabel(
            f"Hours of {date}", fontsize=12
        )
        ax1.set_ylabel("Bids (MWh)", color='green', fontsize=12)
        ax1.tick_params(axis='y', labelcolor='green')
        
        # Create a secondary y-axis for Day-Ahead Prices
        ax2 = ax1.twinx()
        ax2.plot(
            range(1, 25), 
            da_prices[scenario], 
            marker='x', 
            color='blue', 
            label="DA Prices", 
            linestyle='-'
        )
        ax2.set_ylabel("Prices (€)", color='blue', fontsize=12)
        ax2.tick_params(axis='y', labelcolor='blue')
        
        # Add title and grid
        plt.title(f"Scenario {scenario + 1} - Day-Ahead Bids and Prices (Risk Aversion: {risk_aversion}, CVaR: {cvar})", fontsize=14)
        ax1.grid(alpha=0.3)
        
        # Add legends
        ax1.legend(loc="upper left")
        ax2.legend(loc="upper right")
        
        # Tighten layout and show the plot
        plt.tight_layout()
        plt.show()

In [5]:
def plot_accumulated_bids(DA_bids_optimal, da_prices, date, risk_aversion=0, cvar=0.95):
    """
    Plot accumulated bids and prices for all scenarios.

    :param DA_bids_optimal: A 2D array of optimal Day-Ahead bids (shape: [scenarios, hours]).
    :param da_prices: A 2D array of Day-Ahead prices for each scenario (shape: [scenarios, hours]).
    :param date: The date for which the plot is created (string format 'YYYY-MM-DD').
    :param risk_aversion: The risk aversion coefficient used in optimization.
    :param cvar: The CVaR level used in optimization.
    """
    # Create the plot with a secondary y-axis
    plt.figure(figsize=(12, 6))

    # Plot bids for each scenario with open green triangles and red for zero bids
    for scenario in range(DA_bids_optimal.shape[0]):
        for hour in range(DA_bids_optimal.shape[1]):
            if DA_bids_optimal[scenario, hour] == 0:
                plt.scatter(hour + 1, DA_bids_optimal[scenario, hour],
                            edgecolor='red', facecolor='none', marker='^', s=80,
                            label=f"Scenario {scenario + 1} Zero Bid" if hour == 0 else "", alpha=0.8)
            else:
                plt.scatter(hour + 1, DA_bids_optimal[scenario, hour],
                            edgecolor='green', facecolor='none', marker='^', s=80,
                            label=f"Scenario {scenario + 1} Bid" if hour == 0 else "", alpha=0.8)

    # Add the day-ahead prices for each scenario on the secondary y-axis
    ax = plt.gca()
    ax2 = ax.twinx()
    for scenario in range(DA_bids_optimal.shape[0]):
        ax2.plot(range(1, 25), da_prices[scenario, :],
                 marker='x', linestyle='--', label=f"Scenario {scenario + 1} Prices", alpha=0.7)

    # Set labels, titles, and legends
    ax.set_xlabel(f"Hours of {date}", fontsize=12)
    ax.set_ylabel("Bid Quantity (MWh)", color='tab:green', fontsize=12)
    ax2.set_ylabel("Day-Ahead Price (€)", color='tab:orange', fontsize=12)
    plt.title(f"Bid Ladder with Quantities and Prices Across Scenarios (Risk Aversion: {risk_aversion}, CVaR: {cvar})", fontsize=14)

    # Style the grid
    ax.grid(True, linestyle='--', alpha=0.5)

    # Adjust layout and show the plot
    plt.tight_layout()
    plt.show()

In [6]:
def plot_schedule(
    schedule, date, risk_aversion=0, cvar=0.95
):
    # Plot
    fig, ax1 = plt.subplots(figsize=(16, 4))
    
    # Primary axis (Volume Risk and DA Commitment)
    bar_width = 0.4  # Adjust bar width for better clarity
    hours = schedule['Hour']
    
    # Add bars for volume risk and market volume risk
    ax1.bar(hours - bar_width / 2, schedule['volume_risk'], alpha=0.4, label='Volume Risk (MWh)', color='orange', width=bar_width)
    ax1.bar(hours + bar_width / 2, schedule['market_volume_risk'], alpha=0.4, label='Market Volume Risk (MWh)', color='cyan', width=bar_width)
    
    # Add scatter for DA Commitment
    ax1.scatter(schedule['Hour'], schedule['DA_Bid'], label='DA Commitment (MWh)', edgecolor='green', facecolor='none', marker='^', s=100)
    
    # Add dotted line for forecasted generation
    ax1.plot(schedule['Hour'], schedule['forecast_gen'], linestyle='--', color='red', label='Forecasted Generation (MWh)', linewidth=2)
    
    # Configure primary axis
    ax1.set_xlabel(f"Hours of {date}", fontsize=12)
    ax1.set_ylabel('Volume (MWh)', fontsize=14)
    ax1.tick_params(axis='y', labelsize=12)
    ax1.set_xticks(schedule['Hour'])
    ax1.tick_params(axis='x', labelsize=12)
    ax1.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax1.legend(loc='upper left', fontsize=12)
    
    # Secondary axis (MCP)
    ax2 = ax1.twinx()
    ax2.plot(schedule['Hour'], schedule['MCP_DA'], marker='o', linestyle='-', label='DA (€)', color='purple', markersize=7)
    ax2.plot(schedule['Hour'], schedule['MCP_ID'], marker='o', linestyle='-', label='ID (€)', color='blue', markersize=7)
    ax2.set_ylabel('MCP (€)', fontsize=14)
    ax2.tick_params(axis='y', labelsize=12)
    ax2.legend(loc='upper right', fontsize=12)
    
    # Title and grid
    plt.title(f'DA Commitment, MCP, Volume Risk, Market Volume Risk, and Forecasted Generation Per Hour (Risk Aversion: {risk_aversion}, CVaR: {cvar}', fontsize=16)
    ax1.grid(alpha=0.3)
    
    # Tight layout and show plot
    plt.tight_layout()
    plt.show()


In [7]:
def prepare_schedule(date, results, da_prices_path, mcp_da_path, mcp_id_path, forecast_path, actual_path):
    """
    Prepare the schedule DataFrame for a given date.

    :param date: The date for which the schedule is prepared (string format 'YYYY-MM-DD').
    :param results: Dictionary containing DA bids for each date.
    :param da_prices_path: Path pattern for Day-Ahead prices CSV files.
    :param mcp_da_path: Path pattern for MCP Day-Ahead CSV files.
    :param mcp_id_path: Path pattern for MCP Intraday CSV files.
    :param forecast_path: Path to forecasted generation CSV file.
    :param actual_path: Path to actual production CSV file.
    :return: A DataFrame containing the schedule for the specified date.
    """
    # Prepare data for bids
    data = []
    da_prices = pd.read_csv(da_prices_path.format(date=date)).iloc[:, 1:].to_numpy()
    for scenario in range(5):
        for hour in range(24):
            data.append({
                "Scenario": scenario + 1,
                "Hour": hour,
                "DA_Bid": results[date]['DA_bids'][scenario, hour],
                "DA_Price": da_prices[scenario, hour]
            })
    df = pd.DataFrame(data)

    # Load MCP
    mcp_da = pd.read_csv(mcp_da_path.format(date=date))
    mcp_id = pd.read_csv(mcp_id_path.format(date=date))

    # Preprocess MCP data
    mcp_da['Hour'] = pd.to_datetime(mcp_da['delivery_begin']).dt.hour
    mcp_da.rename(columns={'DA_price': 'MCP_DA'}, inplace=True)
    mcp_id['Hour'] = pd.to_datetime(mcp_id['delivery_begin']).dt.hour
    mcp_id.rename(columns={'ID3_price': 'MCP_ID'}, inplace=True)
    # df['Hour'] -= 1

    # Merge DataFrames
    merged = pd.merge(pd.merge(df, mcp_da, on='Hour', how='inner'), mcp_id, on='Hour', how='inner')
    
    # Filter and aggregate
    filtered = merged[merged['DA_Price'] <= merged['MCP_DA']]
    aggregated = filtered.groupby('Hour', as_index=False).agg(
        DA_Bid=('DA_Bid', 'sum'),
        MCP_DA=('MCP_DA', 'first'),
        MCP_ID=('MCP_ID', 'first')
    )

    # Ensure all hours are included and fill missing values
    schedule = pd.DataFrame({'Hour': range(24)}).merge(aggregated, on='Hour', how='left')
    schedule.fillna({'DA_Bid': 0, 'MCP_DA': 0, 'MCP_ID': 0}, inplace=True)

    # Add forecasted generation and actual production
    forecasted_gen = pd.read_csv(forecast_path, index_col=0).loc[f"{date} 00:00:00":f"{date} 23:30:00", :]['frozen_fc_da'].values
    actual_prod = pd.read_csv(actual_path, index_col=0).loc[f"{date} 00:00:00":f"{date} 23:30:00", :]['rt_signal'].values
    schedule['forecast_gen'] = forecasted_gen
    schedule['actual_prod'] = actual_prod
    schedule['volume_risk'] = schedule['actual_prod'] - schedule['DA_Bid']
    schedule['market_volume_risk'] = schedule['actual_prod'] - schedule['forecast_gen']

    return schedule

## Run the Optimizer with Lambda = 1, CVaR = 0.95

In [8]:
# Example usage
lamb = 1
alpha = 0.95
optimizer_1 = BiddingOptimization(limit_price=-2, lambda_risk_aversion=lamb, alpha_cvar=alpha)
results_1 = optimizer_1.run(dates);

Running optimization for 2024-09-17...


FileNotFoundError: [Errno 2] No such file or directory: 'reduced_scenarios/da/scenario_2024-09-17.csv'

## Run the Optimizer with Lambda = 0.5, CVaR = 0.95

In [ ]:
lamb = 0.5
alpha = 0.95
optimizer_05 = BiddingOptimization(limit_price=-2, lambda_risk_aversion=lamb, alpha_cvar=alpha)
results_05 = optimizer_05.run(dates);

## Run the Optimizer with Lambda = 0, CVaR = 0.95

In [ ]:
optimizer_00 = BiddingOptimization(limit_price=-2, lambda_risk_aversion=lamb, alpha_cvar=alpha)
results_00 = optimizer_00.run(dates);

## Bidding Across Scenarios Plotting

### Bidding for risk_aversion = 1

In [ ]:
# Call the function
for date in dates:
    plot_bidding(
        DA_bids_optimal=results_1[date]['DA_bids'],
        da_prices=pd.read_csv(f"reduced_scenarios/da/scenario_{date}.csv").iloc[:, 1:].to_numpy(),
        date=date,
        risk_aversion=1,
        cvar=alpha
    )

### Bidding for risk_aversion = 0.5

In [ ]:
# Call the function
for date in dates:
    plot_bidding(
        DA_bids_optimal=results_05[date]['DA_bids'],
        da_prices=pd.read_csv(f"reduced_scenarios/da/scenario_{date}.csv").iloc[:, 1:].to_numpy(),
        date=date,
        risk_aversion=0.5,
        cvar=alpha
    )

### Bidding for risk_aversion = 0

In [ ]:
# Call the function
for date in dates:
    plot_bidding(
        DA_bids_optimal=results_00[date]['DA_bids'],
        da_prices=pd.read_csv(f"reduced_scenarios/da/scenario_{date}.csv").iloc[:, 1:].to_numpy(),
        date=date,
        risk_aversion=0,
        cvar=alpha
    )

## Accumulated Bids Plotting

### Accumulated Bids for risk_aversion = 0

In [ ]:
# Loop through each date and plot accumulated bids
for date, data in results_1.items():
    DA_bids_optimal = results_1[date]['DA_bids']
    plot_accumulated_bids(
        DA_bids_optimal=DA_bids_optimal,
        da_prices=pd.read_csv(f"reduced_scenarios/da/scenario_{date}.csv").iloc[:, 1:].to_numpy(),
        date=date,
        risk_aversion=1,
        cvar=0.95
    )

### Accumulated Bids for risk_aversion = 0.5

In [ ]:
# Loop through each date and plot accumulated bids
for date, data in results_05.items():
    DA_bids_optimal = results_05[date]['DA_bids']
    plot_accumulated_bids(
        DA_bids_optimal=DA_bids_optimal,
        da_prices=pd.read_csv(f"reduced_scenarios/da/scenario_{date}.csv").iloc[:, 1:].to_numpy(),
        date=date,
        risk_aversion=1,
        cvar=0.95
    )

### Accumulated Bids for risk_aversion = 0

In [ ]:
# Loop through each date and plot accumulated bids
for date, data in results_00.items():
    DA_bids_optimal = results_00[date]['DA_bids']
    plot_accumulated_bids(
        DA_bids_optimal=DA_bids_optimal,
        da_prices=pd.read_csv(f"reduced_scenarios/da/scenario_{date}.csv").iloc[:, 1:].to_numpy(),
        date=date,
        risk_aversion=1,
        cvar=0.95
    )

## Day-Ahead Schedule Plotting

### Day-Ahead Schedule risk_aversion = 1

In [ ]:
# Optimized loop to call plot_schedule
schedule_1 = {}
for date in dates:
    schedule_1[date] = prepare_schedule(
        date=date,
        results=results_1,
        da_prices_path=f"reduced_scenarios/da/scenario_{date}.csv",
        mcp_da_path=f"scenarios/da/prices_{date}.csv",
        mcp_id_path=f"scenarios/id/prices_{date}.csv",
        forecast_path="data/HKZ_forecast_da.csv",
        actual_path="data/HKZ_actual_prod.csv"
    )
    # save schedule
    schedule_1[date].to_csv(f'strategy/optim1/01/schedule_{date}')
    plot_schedule(schedule_1[date], date, risk_aversion=1, cvar=0.95)

### Day-Ahead Schedule risk_aversion = 0.5

In [ ]:
# Optimized loop to call plot_schedule
schedule_05 = {}
for date in dates:
    schedule_05[date] = prepare_schedule(
        date=date,
        results=results_05,
        da_prices_path=f"reduced_scenarios/da/scenario_{date}.csv",
        mcp_da_path=f"scenarios/da/prices_{date}.csv",
        mcp_id_path=f"scenarios/id/prices_{date}.csv",
        forecast_path="data/HKZ_forecast_da.csv",
        actual_path="data/HKZ_actual_prod.csv"
    )
    # save schedule
    schedule_05[date].to_csv(f'strategy/optim1/05/schedule_{date}')
    plot_schedule(schedule_05[date], date, risk_aversion=0.5, cvar=0.95)

### Day-Ahead Schedule risk_aversion = 0

In [ ]:
# Optimized loop to call plot_schedule
schedule_00 = {}
for date in dates:
    schedule_00[date] = prepare_schedule(
        date=date,
        results=results_00,
        da_prices_path=f"reduced_scenarios/da/scenario_{date}.csv",
        mcp_da_path=f"scenarios/da/prices_{date}.csv",
        mcp_id_path=f"scenarios/id/prices_{date}.csv",
        forecast_path="data/HKZ_forecast_da.csv",
        actual_path="data/HKZ_actual_prod.csv"
    )
    # save schedule
    schedule_00[date].to_csv(f'strategy/optim1/00/schedule_{date}')
    plot_schedule(schedule_00[date], date, risk_aversion=0, cvar=0.95)